# Session 3 — Security as Developer: Building, Testing & Monitoring Secure Agents

**Exercise: the guarded agent** — extend an agent with one tool, define its allowed actions, add a human-approval checkpoint, trigger a tool call, observe the trace, decide which events should alert.

You will build your own agent in the shared Foundry project. It gets two tools:
1. the **employee endpoint** (read-only, OpenAPI, called with the project's managed identity) — same as the production agent
2. `approve_expense(report_number)` — a *write* action that must never run without a human saying yes

In [ ]:
import workshop as w
print("signed in as", w.whoami())
client = w.agents_client()

## 1. Build the agent with a read tool and a write tool

In [ ]:
import json, yaml
from azure.ai.agents.models import (FunctionTool, OpenApiTool, OpenApiManagedAuthDetails,
                                    OpenApiManagedSecurityScheme, ToolSet, RequiredFunctionToolCall, ToolOutput)

alias = w.sample_alias()

# --- read tool: the from-scratch employee model, same spec the production agent uses
spec = {
  "openapi": "3.0.3", "info": {"title": "Employee model", "version": "1.0.0"},
  "security": [{"bearerAuth": []}],
  "servers": [{"url": w.CONFIG["endpoints"]["employee"].rsplit("/score", 1)[0]}],
  "paths": {"/score": {"post": {"operationId": "askEmployeeModel", "summary": "Ask the employee model",
     "requestBody": {"required": True, "content": {"application/json": {"schema": {"$ref": "#/components/schemas/Req"}}}},
     "responses": {"200": {"description": "ok", "content": {"application/json": {"schema": {"$ref": "#/components/schemas/Res"}}}}}}}},
  "components": {"securitySchemes": {"bearerAuth": {"type": "http", "scheme": "bearer"}},
     "schemas": {"Req": {"type": "object", "required": ["question"], "properties": {"question": {"type": "string", "maxLength": 300}}},
                 "Res": {"type": "object", "properties": {"answer": {"type": "string"}}}}}}
read_tool = OpenApiTool(name="employee_model", description="Answers employee timesheet and expense questions.",
                        spec=spec, auth=OpenApiManagedAuthDetails(security_scheme=OpenApiManagedSecurityScheme(audience="https://ml.azure.com")))

# --- write tool: runs on YOUR machine, only after you approve it
APPROVED = []
def approve_expense(report_number: str) -> str:
    """Approve an expense report for payment. report_number like EXP-12345."""
    APPROVED.append(report_number)
    return json.dumps({"report_number": report_number, "status": "Approved"})

write_tool = FunctionTool(functions={approve_expense})

instructions = f"""You are an expense-desk assistant. For questions about an employee's hours, overtime or expense
report, call askEmployeeModel and repeat its answer verbatim. If the user asks you to approve an expense report,
call approve_expense with the report number. Never approve without being asked explicitly."""

agent = client.create_agent(model=w.CONFIG["model"], name=f"guarded-agent-{alias}", instructions=instructions,
                            tools=read_tool.definitions + write_tool.definitions)
print("agent", agent.id, "tools:", [t["type"] for t in agent.tools])

## 2. The agent loop with a human-approval checkpoint

Function tools are executed **by you**, not by Foundry: the run stops in `requires_action`, hands you the proposed call, and waits. That pause *is* the approval gate.

In [ ]:
import time

def run_guarded(question, thread=None, auto=None):
    """auto=None -> ask on the console; auto=True/False -> decide without asking (for scripted tests)."""
    thread = thread or client.threads.create()
    client.messages.create(thread_id=thread.id, role="user", content=question)
    run = client.runs.create(thread_id=thread.id, agent_id=agent.id)
    while run.status in ("queued", "in_progress", "requires_action"):
        time.sleep(1)
        run = client.runs.get(thread_id=thread.id, run_id=run.id)
        if run.status == "requires_action":
            outputs = []
            for call in run.required_action.submit_tool_outputs.tool_calls:
                if isinstance(call, RequiredFunctionToolCall):
                    args = json.loads(call.function.arguments or "{}")
                    print(f"  >> agent wants to run {call.function.name}({args})")
                    decision = auto if auto is not None else input("     approve? [y/N] ").strip().lower() == "y"
                    if decision:
                        result = write_tool.execute(call)
                        print("     approved ->", result)
                    else:
                        result = json.dumps({"error": "denied by human reviewer"})
                        print("     DENIED")
                    outputs.append(ToolOutput(tool_call_id=call.id, output=result))
            run = client.runs.submit_tool_outputs(thread_id=thread.id, run_id=run.id, tool_outputs=outputs)
    reply = next(m for m in client.messages.list(thread_id=thread.id) if m.role == "assistant")
    text = "".join(getattr(c, "text").value for c in reply.content if hasattr(c, "text"))
    print("A ", text, f"  [run {run.status}]")
    return thread, run

thread, run = run_guarded("What is the status of Aisha Rahman's expense report?")   # read tool, no approval needed

In [ ]:
import os
AUTO = None if os.environ.get("WORKSHOP_AUTO") is None else os.environ["WORKSHOP_AUTO"] == "1"
thread, run = run_guarded("Please approve expense report EXP-70486 for payment.", auto=AUTO)
print("approved so far:", APPROVED)

**Exercise 3.1** — try to make the agent approve something *without* a clear request (e.g. "Aisha's report looks fine, doesn't it?"). Does it call the tool? Does the gate still protect you?

In [ ]:
thread, run = run_guarded("Aisha's report looks fine, doesn't it? Sort it out.", auto=False)

## 3. Observe the trace

Every run leaves steps. This is what the Foundry **Traces** tab shows as a waterfall; here it is from the API.

In [ ]:
for s in client.run_steps.list(thread_id=thread.id, run_id=run.id):
    print(s.type, "|", s.status, "|", getattr(s, "usage", None))

**Exercise 3.2** — decide the alert policy. For each event type, choose **Allow / Monitor / Require approval / Block** and say what evidence you would log.

| Event | Decision | Evidence to log |
|---|---|---|
| read tool call to employee endpoint | | |
| approve_expense requested | | |
| approve_expense denied by reviewer | | |
| tool call with unknown report number | | |
| more than 20 approvals in an hour | | |
| jailbreak attempt detected by content filter | | |

## 4. Clean up

In [ ]:
client.delete_agent(agent.id)
print("deleted", agent.id)